<a href="https://colab.research.google.com/github/wnstj1126-debug/-/blob/main/M0_AUDIT_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M0_AUDIT_v1 — DeepBind FEMTO M0 Baseline 독립 감사 노트북

---

| 항목 | 내용 |
|:---|:---|
| **노트북** | M0_AUDIT_v1 |
| **목적** | M0 frozen 데이터 계보·재현성·reg_FAR 의미 감사 |
| **기준 모델** | `M0_FEMTO_Baseline_v1_baseline고정.ipynb` |
| **모델 변경** | 없음 |
| **원본 변경** | 없음 |
| **M1/M2 구현** | 없음 |
| **OnlineSim feature parity 비교** | **제외** |

---

## 감사 항목

1. FEMTO Training/Test/Validation 데이터 출처 및 레코드 수 감사
2. Test와 Validation의 동일 bearing 간 prefix/truncated 관계 감사
3. 중복 bearing 이름 발생 원인 확인
4. M0 frozen 결과의 재현성 확인
5. 기존 `reg_FAR`의 정확한 계산 의미 확인
6. `alarm_idx`의 시작 시점과 경보 확정 시점 구분
7. frozen JSON과 실제 결과 및 pass criteria 간 일관성 검사

## 감사 범위에서 제외

- `OnlineSim_FEMTO_v3.ipynb`와 frozen M0의 특징 함수 일치 여부
- OnlineSim 코드와 frozen M0 코드의 feature parity
- M1/M2 모델 구현
- 2048 frame bridge 모델 구현
- 특징 정의 변경, threshold/K 변경, pass criteria 변경
- 기존 결과를 개선하기 위한 파라미터 튜닝

> **주의:** `"OnlineSim_FEMTO_v3의 9특징과 동일 구현"` 문구는 복사·붙여넣기 실수로 처리하며 감사 대상에서 제외한다.

## Read-only 원칙

쓰기 작업은 `audit_outputs/YYYYMMDD_HHMMSS/` 폴더 안에서만 허용된다.
다음 파일은 절대 수정하지 않는다:
- `M0_FEMTO_Baseline_v1.ipynb`
- `M0_FEMTO_Baseline_v1_baseline고정.ipynb`
- `OnlineSim_FEMTO_v3.ipynb`
- `BASELINE_M0_frozen.json`
- `m0_baseline_result.csv`
- `CLAUDE.md`
- FEMTO 원본 CSV 전체

In [1]:
# ============================================================
# 셀 01 — 설정
# ============================================================

from pathlib import Path

AUDIT_VERSION = "M0_AUDIT_v1"

# FEMTO 샘플링 파라미터
FEMTO_FS = 25600                    # Hz
FEMTO_SNAPSHOT_DURATION_SEC = 0.1   # 스냅샷 1개 = 0.1초  ← 절대 혼동 금지
FEMTO_DECISION_INTERVAL_SEC = 10.0  # 의사결정 간격 = 10초 ← 절대 혼동 금지

# M0 고정 파라미터
COL_H = 4    # H축 (수평 방향 진동) 컬럼 인덱스
K     = 6.0  # Mahalanobis 임계 배수
CONSEC = 5   # 연속 초과 조건
REG_MIN = 90
REG_MAX = 600

# 감사 옵션
FULL_BYTE_HASH = True
SEMANTIC_HASH_ON_BYTE_MISMATCH = True
RECOMPUTE_LEARNING_M0 = True
RECOMPUTE_NON_LEARNING_M0 = False

# 기대 레코드 수
EXPECTED_LEARNING_RECORDS   = 6
EXPECTED_TEST_RECORDS       = 11
EXPECTED_FULL_TEST_RECORDS  = 11
EXPECTED_TOTAL_RECORDS      = 28

# Frozen 일관성 허용 오차
ABS_TOL      = 1e-6
ROUNDING_TOL = 0.005

# ── 단위 테스트 (assert) ────────────────────────────────────
assert CONSEC == 5,                          f"CONSEC={CONSEC} ≠ 5"
assert FEMTO_DECISION_INTERVAL_SEC == 10.0,  f"FEMTO_DECISION_INTERVAL_SEC={FEMTO_DECISION_INTERVAL_SEC} ≠ 10.0"
assert FEMTO_SNAPSHOT_DURATION_SEC == 0.1,   f"FEMTO_SNAPSHOT_DURATION_SEC={FEMTO_SNAPSHOT_DURATION_SEC} ≠ 0.1"
assert COL_H == 4,                           f"COL_H={COL_H} ≠ 4"
assert K == 6.0,                             f"K={K} ≠ 6.0"

print(f"[CONFIG] {AUDIT_VERSION} 설정 완료")
print(f"  FEMTO_FS                    = {FEMTO_FS} Hz")
print(f"  FEMTO_SNAPSHOT_DURATION_SEC = {FEMTO_SNAPSHOT_DURATION_SEC} s  (스냅샷 길이)")
print(f"  FEMTO_DECISION_INTERVAL_SEC = {FEMTO_DECISION_INTERVAL_SEC} s  (의사결정 간격)")
print(f"  COL_H={COL_H}, K={K}, CONSEC={CONSEC}, REG_MIN={REG_MIN}, REG_MAX={REG_MAX}")
print("[CONFIG] 모든 assert 통과 ✅")

[CONFIG] M0_AUDIT_v1 설정 완료
  FEMTO_FS                    = 25600 Hz
  FEMTO_SNAPSHOT_DURATION_SEC = 0.1 s  (스냅샷 길이)
  FEMTO_DECISION_INTERVAL_SEC = 10.0 s  (의사결정 간격)
  COL_H=4, K=6.0, CONSEC=5, REG_MIN=90, REG_MAX=600
[CONFIG] 모든 assert 통과 ✅


In [2]:
# ============================================================
# 셀 01b — 단위 테스트 (alarm 시퀀스, natural sort)
# ============================================================

import re
import numpy as np

# ── 경보 시퀀스 테스트 ─────────────────────────────────────
def _test_alarm_sequence():
    example = [False, True, False, True, True, True, True, True, False]
    # 기대값:
    #   단일 초과 수 = 6
    #   최대 연속 초과 = 5
    #   alarm event 수 = 1
    #   alarm start index = 3
    #   alarm confirm index = 7
    arr = np.array(example)
    single_count = arr.sum()           # 6
    assert single_count == 6, f"단일 초과 수={single_count} ≠ 6"

    max_consec = 0
    cur = 0
    for v in arr:
        cur = cur + 1 if v else 0
        max_consec = max(max_consec, cur)
    assert max_consec == 5, f"최대 연속 초과={max_consec} ≠ 5"

    # alarm start / confirm index
    consec = CONSEC
    alarm_start = None
    alarm_confirm = None
    alarm_events = 0
    in_alarm = False
    run = 0
    run_start = None
    for i, v in enumerate(arr):
        if v:
            if run == 0:
                run_start = i
            run += 1
            if run == consec and not in_alarm:
                alarm_start = run_start
                alarm_confirm = i
                alarm_events += 1
                in_alarm = True
        else:
            run = 0
            in_alarm = False

    assert alarm_events == 1,   f"alarm event 수={alarm_events} ≠ 1"
    assert alarm_start == 3,    f"alarm_start_idx={alarm_start} ≠ 3"
    assert alarm_confirm == 7,  f"alarm_confirm_idx={alarm_confirm} ≠ 7"
    assert alarm_confirm == alarm_start + consec - 1, "confirm = start + CONSEC - 1 불일치"
    print("  alarm 시퀀스 테스트: ✅")

_test_alarm_sequence()

# ── Natural sort 테스트 ────────────────────────────────────
def natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]

def _test_natural_sort():
    names = ["acc_10.csv", "acc_2.csv", "acc_1.csv"]
    sorted_names = sorted(names, key=natural_sort_key)
    expected = ["acc_1.csv", "acc_2.csv", "acc_10.csv"]
    assert sorted_names == expected, f"natural sort 실패: {sorted_names}"
    print("  natural sort 테스트: ✅")

_test_natural_sort()
print("[UNIT TESTS] 모든 단위 테스트 통과 ✅")

  alarm 시퀀스 테스트: ✅
  natural sort 테스트: ✅
[UNIT TESTS] 모든 단위 테스트 통과 ✅


In [3]:
# ============================================================
# 셀 02 — Drive 마운트 및 경로 확인
# ============================================================

import os
import re
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── PROJECT_ROOT 후보 탐색 ─────────────────────────────────
PROJECT_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/field_iis3dwb"),
    Path("/content/drive/My Drive/Colab Notebooks/field_iis3dwb"),
]
PROJECT_ROOT = None
for c in PROJECT_ROOT_CANDIDATES:
    if c.exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT_NOT_FOUND: field_iis3dwb 폴더를 찾을 수 없습니다.")

# ── FEMTO_ROOT 후보 탐색 ──────────────────────────────────
FEMTO_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/Colab Notebooks/RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
    Path("/content/drive/My Drive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing"),
]
FEMTO_ROOT = None
for c in FEMTO_ROOT_CANDIDATES:
    if c.exists():
        FEMTO_ROOT = c
        break
if FEMTO_ROOT is None:
    raise FileNotFoundError("FEMTO_ROOT_NOT_FOUND: PRONOSTIA_FEMTO_Bearing 폴더를 찾을 수 없습니다.")

# ── Split 폴더 탐색 ──────────────────────────────────────
SPLIT_DIR_NAMES = {
    "LEARNING": "Training(Learning)_set",
    "TEST":     "Test(Test)_set",
    "FULL_TEST": "Validation(Full_Test)_Set",
}

def resolve_split_dir(root: Path, preferred_name: str, split_key: str) -> Path:
    """정확한 이름 우선, 없으면 대소문자 무시 탐색."""
    exact = root / preferred_name
    if exact.exists():
        return exact
    # 대소문자 무시 탐색
    matches = [d for d in root.iterdir() if d.is_dir()
               and d.name.lower() == preferred_name.lower()]
    if len(matches) == 1:
        print(f"  [{split_key}] 대소문자 무시 대체 경로 사용: {matches[0].name}")
        return matches[0]
    elif len(matches) > 1:
        raise RuntimeError(f"AMBIGUOUS_SPLIT_DIRECTORY: {split_key} 후보 복수 발견: {[m.name for m in matches]}")
    else:
        raise FileNotFoundError(f"SPLIT_DIR_NOT_FOUND: {split_key} ({preferred_name}) 을 찾을 수 없습니다.")

LEARNING_DIR  = resolve_split_dir(FEMTO_ROOT, SPLIT_DIR_NAMES["LEARNING"],  "LEARNING")
TEST_DIR      = resolve_split_dir(FEMTO_ROOT, SPLIT_DIR_NAMES["TEST"],      "TEST")
FULL_TEST_DIR = resolve_split_dir(FEMTO_ROOT, SPLIT_DIR_NAMES["FULL_TEST"], "FULL_TEST")

# ── OUTPUT 폴더 생성 ─────────────────────────────────────
AUDIT_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "audit_outputs" / AUDIT_TS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"FEMTO_ROOT    : {FEMTO_ROOT}")
print(f"LEARNING_DIR  : {LEARNING_DIR}")
print(f"TEST_DIR      : {TEST_DIR}")
print(f"FULL_TEST_DIR : {FULL_TEST_DIR}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")

Mounted at /content/drive
PROJECT_ROOT  : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb
FEMTO_ROOT    : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing
LEARNING_DIR  : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing/Training(Learning)_set
TEST_DIR      : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing/Test(Test)_set
FULL_TEST_DIR : /content/drive/MyDrive/Colab Notebooks/3.RunToFailure_Raw_Data/PRONOSTIA_FEMTO_Bearing/Validation(Full_Test)_Set
OUTPUT_DIR    : /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125


In [4]:
# ============================================================
# 셀 03 — 원본 파일 사전 스냅샷
# ============================================================

import hashlib
import json
from datetime import datetime
from pathlib import Path

SOURCE_FILE_NAMES = [
    "BASELINE_M0_frozen.json",
    "m0_baseline_result.csv",
    "M0_FEMTO_Baseline_v1_baseline고정.ipynb",
]

def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def snap_source_files(label: str) -> dict:
    snap = {}
    for fname in SOURCE_FILE_NAMES:
        found = list(PROJECT_ROOT.rglob(fname))
        if not found:
            snap[fname] = {"status": "MISSING"}
            print(f"  [{label}] {fname}: MISSING")
            continue
        p = found[0]
        stat = p.stat()
        sha  = sha256_of_file(p)
        snap[fname] = {
            "status":        "FOUND",
            "absolute_path": str(p),
            "size_bytes":    stat.st_size,
            "modified_time": datetime.fromtimestamp(stat.st_mtime).isoformat(),
            f"sha256_{label}": sha,
        }
        print(f"  [{label}] {fname}: {stat.st_size:,} bytes  sha256={sha[:16]}...")
    return snap

print("=== 원본 파일 사전 스냅샷 (BEFORE) ===")
SOURCE_SNAP_BEFORE = snap_source_files("before")

# 저장
before_path = OUTPUT_DIR / "source_file_snapshot_before.json"
with open(before_path, "w", encoding="utf-8") as f:
    json.dump(SOURCE_SNAP_BEFORE, f, ensure_ascii=False, indent=2)
print(f"\n저장: {before_path}")

# frozen JSON 또는 baseline CSV 없으면 PARTIAL_AUDIT
AUDIT_COMPLETENESS = "FULL"
for fname in ["BASELINE_M0_frozen.json", "m0_baseline_result.csv"]:
    if SOURCE_SNAP_BEFORE.get(fname, {}).get("status") == "MISSING":
        AUDIT_COMPLETENESS = "PARTIAL_AUDIT_MISSING_REFERENCE"
        print(f"\n⚠️  {fname} 없음 → 재현성 감사 상태: {AUDIT_COMPLETENESS}")
print(f"\n감사 완전성: {AUDIT_COMPLETENESS}")

=== 원본 파일 사전 스냅샷 (BEFORE) ===
  [before] BASELINE_M0_frozen.json: 783 bytes  sha256=a4632ad1b9c51976...
  [before] m0_baseline_result.csv: 1,899 bytes  sha256=63f493e6e3482157...
  [before] M0_FEMTO_Baseline_v1_baseline고정.ipynb: MISSING

저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/source_file_snapshot_before.json

감사 완전성: FULL


In [5]:
# ============================================================
# 셀 04 — Split manifest 생성
# ============================================================

import csv
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

def natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]

def is_contiguous(indices):
    if len(indices) < 2:
        return True
    return all(indices[i+1] - indices[i] == 1 for i in range(len(indices)-1))

def extract_index(fname):
    m = re.search(r'(\d+)', Path(fname).stem)
    return int(m.group(1)) if m else -1

SPLIT_MAP = {
    "LEARNING":  LEARNING_DIR,
    "TEST":      TEST_DIR,
    "FULL_TEST": FULL_TEST_DIR,
}

records = []

for split_key, split_dir in SPLIT_MAP.items():
    bearing_dirs = sorted(
        [d for d in split_dir.iterdir() if d.is_dir() and d.name.startswith("Bearing")],
        key=lambda d: natural_sort_key(d.name)
    )
    for bd in bearing_dirs:
        csv_files = sorted(
            [f for f in bd.iterdir() if f.name.startswith("acc_") and f.suffix == ".csv"],
            key=lambda f: natural_sort_key(f.name)
        )
        if not csv_files:
            continue

        indices = [extract_index(f.name) for f in csv_files]
        total_bytes = sum(f.stat().st_size for f in csv_files)
        file_count  = len(csv_files)

        estimated_observation_hours = (
            (file_count - 1) * FEMTO_DECISION_INTERVAL_SEC
        ) / 3600

        record_uid = f"{split_key}/{bd.name}"

        records.append({
            "record_uid":                  record_uid,
            "split":                       split_key,
            "logical_bearing_id":          bd.name,
            "absolute_directory":          str(bd),
            "relative_directory":          str(bd.relative_to(FEMTO_ROOT)),
            "file_count":                  file_count,
            "first_filename":              csv_files[0].name,
            "last_filename":               csv_files[-1].name,
            "filename_index_min":          min(indices),
            "filename_index_max":          max(indices),
            "filename_index_contiguous":   is_contiguous(sorted(indices)),
            "total_bytes":                 total_bytes,
            "sample_rate_hz":              FEMTO_FS,
            "snapshot_duration_sec":       FEMTO_SNAPSHOT_DURATION_SEC,
            "decision_interval_sec":       FEMTO_DECISION_INTERVAL_SEC,
            "estimated_observation_hours": round(estimated_observation_hours, 4),
        })

MANIFEST_DF = pd.DataFrame(records)

# 저장
manifest_csv  = OUTPUT_DIR / "femto_split_manifest.csv"
manifest_json = OUTPUT_DIR / "femto_split_manifest.json"
MANIFEST_DF.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
with open(manifest_json, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# 집계
cnt = MANIFEST_DF.groupby("split")["record_uid"].count().to_dict()
actual_learning   = cnt.get("LEARNING", 0)
actual_test       = cnt.get("TEST", 0)
actual_full_test  = cnt.get("FULL_TEST", 0)
actual_total      = len(MANIFEST_DF)

manifest_match = (
    actual_learning  == EXPECTED_LEARNING_RECORDS and
    actual_test      == EXPECTED_TEST_RECORDS and
    actual_full_test == EXPECTED_FULL_TEST_RECORDS and
    actual_total     == EXPECTED_TOTAL_RECORDS
)

print("=== Split Manifest ===")
print(f"  LEARNING  실제={actual_learning:2d} / 기대={EXPECTED_LEARNING_RECORDS}")
print(f"  TEST      실제={actual_test:2d} / 기대={EXPECTED_TEST_RECORDS}")
print(f"  FULL_TEST 실제={actual_full_test:2d} / 기대={EXPECTED_FULL_TEST_RECORDS}")
print(f"  TOTAL     실제={actual_total:2d} / 기대={EXPECTED_TOTAL_RECORDS}")
print(f"  manifest_expected_count_match = {manifest_match}")
print(f"\n  저장: {manifest_csv}")
display(MANIFEST_DF[["record_uid","split","logical_bearing_id","file_count","estimated_observation_hours"]])

=== Split Manifest ===
  LEARNING  실제= 6 / 기대=6
  TEST      실제=11 / 기대=11
  FULL_TEST 실제=11 / 기대=11
  TOTAL     실제=28 / 기대=28
  manifest_expected_count_match = True

  저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/femto_split_manifest.csv


,record_uid,split,logical_bearing_id,file_count,estimated_observation_hours
0,LEARNING/Bearing1_1,LEARNING,Bearing1_1,2803,7.7833
1,LEARNING/Bearing1_2,LEARNING,Bearing1_2,871,2.4167
2,LEARNING/Bearing2_1,LEARNING,Bearing2_1,911,2.5278
3,LEARNING/Bearing2_2,LEARNING,Bearing2_2,797,2.2111
4,LEARNING/Bearing3_1,LEARNING,Bearing3_1,515,1.4278
5,LEARNING/Bearing3_2,LEARNING,Bearing3_2,1637,4.5444
6,TEST/Bearing1_3,TEST,Bearing1_3,590,1.6361
7,TEST/Bearing1_4,TEST,Bearing1_4,1139,3.1611
8,TEST/Bearing1_5,TEST,Bearing1_5,2302,6.3917
9,TEST/Bearing1_6,TEST,Bearing1_6,2232,6.1972


In [6]:
# ============================================================
# 셀 05 — 중복 이름 감사
# ============================================================

import pandas as pd

dup_records = []

grouped = MANIFEST_DF.groupby("logical_bearing_id")

for bid, grp in grouped:
    splits = grp["split"].tolist()
    n_splits = len(splits)

    if n_splits == 1:
        category = "UNIQUE"
    else:
        # 같은 split 안에 두 번 있는지 확인
        if len(splits) != len(set(splits)):
            category = "SAME_SPLIT_DUPLICATE_DIRECTORY"
        else:
            category = "CROSS_SPLIT_SAME_LOGICAL_ID"

    dup_records.append({
        "logical_bearing_id": bid,
        "count":              n_splits,
        "splits":             ", ".join(sorted(splits)),
        "category":           category,
    })

DUP_DF = pd.DataFrame(dup_records)
dup_csv = OUTPUT_DIR / "duplicate_name_audit.csv"
DUP_DF.to_csv(dup_csv, index=False, encoding="utf-8-sig")

cross_split = DUP_DF[DUP_DF["category"] == "CROSS_SPLIT_SAME_LOGICAL_ID"]
same_split  = DUP_DF[DUP_DF["category"] == "SAME_SPLIT_DUPLICATE_DIRECTORY"]

print("=== 중복 이름 감사 ===")
print(f"  CROSS_SPLIT_SAME_LOGICAL_ID    : {len(cross_split)}개")
print(f"  SAME_SPLIT_DUPLICATE_DIRECTORY : {len(same_split)}개")
print(f"  UNIQUE                         : {len(DUP_DF[DUP_DF['category']=='UNIQUE'])}개")

print()
print("[근본 원인]")
print("기존 M0 코드는 전체 경로가 아니라 bd.name을 결과 ID로 사용했기 때문에")
print("TEST/BearingX_Y와 FULL_TEST/BearingX_Y가 동일 이름으로 표시됐다.")

# TEST ↔ FULL_TEST 공통 bearing 확인
test_ids      = set(MANIFEST_DF[MANIFEST_DF["split"]=="TEST"]["logical_bearing_id"])
full_test_ids = set(MANIFEST_DF[MANIFEST_DF["split"]=="FULL_TEST"]["logical_bearing_id"])
common_ids    = sorted(test_ids & full_test_ids)
print(f"\nTEST ∩ FULL_TEST 공통 bearing ({len(common_ids)}개): {common_ids}")

print(f"\n저장: {dup_csv}")
display(DUP_DF)

=== 중복 이름 감사 ===
  CROSS_SPLIT_SAME_LOGICAL_ID    : 11개
  SAME_SPLIT_DUPLICATE_DIRECTORY : 0개
  UNIQUE                         : 6개

[근본 원인]
기존 M0 코드는 전체 경로가 아니라 bd.name을 결과 ID로 사용했기 때문에
TEST/BearingX_Y와 FULL_TEST/BearingX_Y가 동일 이름으로 표시됐다.

TEST ∩ FULL_TEST 공통 bearing (11개): ['Bearing1_3', 'Bearing1_4', 'Bearing1_5', 'Bearing1_6', 'Bearing1_7', 'Bearing2_3', 'Bearing2_4', 'Bearing2_5', 'Bearing2_6', 'Bearing2_7', 'Bearing3_3']

저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/duplicate_name_audit.csv


,logical_bearing_id,count,splits,category
0,Bearing1_1,1,LEARNING,UNIQUE
1,Bearing1_2,1,LEARNING,UNIQUE
2,Bearing1_3,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID
3,Bearing1_4,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID
4,Bearing1_5,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID
5,Bearing1_6,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID
6,Bearing1_7,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID
7,Bearing2_1,1,LEARNING,UNIQUE
8,Bearing2_2,1,LEARNING,UNIQUE
9,Bearing2_3,2,"FULL_TEST, TEST",CROSS_SPLIT_SAME_LOGICAL_ID


In [7]:
# ============================================================
# 셀 06 — TEST/FULL_TEST prefix 감사
# ============================================================

import hashlib
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

EXPECTED_COMMON_BEARINGS = [
    "Bearing1_3", "Bearing1_4", "Bearing1_5", "Bearing1_6", "Bearing1_7",
    "Bearing2_3", "Bearing2_4", "Bearing2_5", "Bearing2_6", "Bearing2_7",
    "Bearing3_3",
]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def get_sorted_acc_files(bearing_dir: Path):
    files = [f for f in bearing_dir.iterdir()
             if f.name.startswith("acc_") and f.suffix == ".csv"]
    return sorted(files, key=lambda f: natural_sort_key(f.name))

def classify_relation(test_count, full_test_count, byte_equal_count):
    """파일 수와 hash 일치 수로 관계 판정."""
    shorter = min(test_count, full_test_count)
    if test_count == full_test_count:
        if byte_equal_count == test_count:
            return "EXACT_SAME_SEQUENCE"
        else:
            return "SAME_LENGTH_DIFFERENT_CONTENT"
    elif test_count < full_test_count:
        if byte_equal_count == test_count:
            return "TEST_IS_PREFIX_OF_FULL_TEST"
        elif byte_equal_count > 0:
            return "PARTIAL_PREFIX"
        else:
            return "DIFFERENT_SEQUENCE"
    else:  # full_test_count < test_count
        if byte_equal_count == full_test_count:
            return "FULL_TEST_IS_PREFIX_OF_TEST"
        elif byte_equal_count > 0:
            return "PARTIAL_PREFIX"
        else:
            return "DIFFERENT_SEQUENCE"

def semantic_compare(path_a: Path, path_b: Path) -> bool:
    """CSV 값 레벨 비교 (COL_H, COL_H+1 기준)."""
    try:
        a = np.loadtxt(path_a, delimiter=",")
        b = np.loadtxt(path_b, delimiter=",")
        if a.shape != b.shape:
            return False
        cols = min(a.shape[1], COL_H + 2) if a.ndim == 2 else 1
        if a.ndim == 2 and cols >= COL_H + 2:
            return np.allclose(a[:, COL_H:COL_H+2], b[:, COL_H:COL_H+2],
                               rtol=1e-10, atol=1e-12, equal_nan=True)
        return np.allclose(a, b, rtol=1e-10, atol=1e-12, equal_nan=True)
    except Exception:
        return False

# ── 체크포인트 로드 ───────────────────────────────────────
checkpoint_path = OUTPUT_DIR / "prefix_audit_checkpoint.csv"
done_bearings = set()
audit_rows = []
if checkpoint_path.exists():
    chk_df = pd.read_csv(checkpoint_path)
    done_bearings = set(chk_df["logical_bearing_id"].tolist())
    audit_rows = chk_df.to_dict("records")
    print(f"체크포인트 로드: {len(done_bearings)}개 이미 완료")

# ── 비교 수행 ───────────────────────────────────────────
file_comparison_rows = []

for i, bid in enumerate(EXPECTED_COMMON_BEARINGS):
    if bid in done_bearings:
        print(f"  [{i+1}/{len(EXPECTED_COMMON_BEARINGS)}] {bid}: 이미 완료 (건너뜀)")
        continue

    test_dir      = TEST_DIR      / bid
    full_test_dir = FULL_TEST_DIR / bid

    if not test_dir.exists() or not full_test_dir.exists():
        audit_rows.append({
            "logical_bearing_id": bid, "relation": "UNRESOLVED",
            "reason": "디렉토리 없음"
        })
        continue

    test_files      = get_sorted_acc_files(test_dir)
    full_test_files = get_sorted_acc_files(full_test_dir)
    test_count      = len(test_files)
    full_test_count = len(full_test_files)
    shorter         = "TEST" if test_count <= full_test_count else "FULL_TEST"
    aligned_count   = min(test_count, full_test_count)

    print(f"  [{i+1}/{len(EXPECTED_COMMON_BEARINGS)}] {bid}: TEST={test_count}, FULL_TEST={full_test_count}")

    # SHA-256 비교
    byte_equal_count   = 0
    first_mismatch_idx = None
    mismatch_pairs     = []

    if FULL_BYTE_HASH:
        for j, (tf, ff) in enumerate(zip(test_files[:aligned_count],
                                          full_test_files[:aligned_count])):
            if (j + 1) % 500 == 0:
                print(f"    [{i+1}/{len(EXPECTED_COMMON_BEARINGS)}] {bid}: {j+1}/{aligned_count}")
            h_test = sha256_file(tf)
            h_full = sha256_file(ff)
            if h_test == h_full:
                byte_equal_count += 1
            else:
                if first_mismatch_idx is None:
                    first_mismatch_idx = j
                mismatch_pairs.append((j, tf, ff))

    byte_equal_ratio = byte_equal_count / aligned_count if aligned_count > 0 else 0.0

    # Semantic hash (byte mismatch 있는 경우)
    semantic_result = "N/A"
    if SEMANTIC_HASH_ON_BYTE_MISMATCH and mismatch_pairs:
        sample_indices = [0, len(mismatch_pairs)//2, len(mismatch_pairs)-1]
        sample_pairs   = [mismatch_pairs[k] for k in sorted(set(sample_indices))
                          if k < len(mismatch_pairs)]
        sample_equal   = all(semantic_compare(p[1], p[2]) for p in sample_pairs)
        if sample_equal and len(mismatch_pairs) > 3:
            all_equal = all(semantic_compare(p[1], p[2]) for p in mismatch_pairs)
            semantic_result = "SEMANTIC_EQUAL" if all_equal else "SEMANTIC_DIFFERENT"
        elif sample_equal:
            semantic_result = "SEMANTIC_EQUAL_SAMPLE"
        else:
            semantic_result = "SEMANTIC_DIFFERENT"

    relation = classify_relation(test_count, full_test_count, byte_equal_count)

    row = {
        "logical_bearing_id":      bid,
        "test_file_count":         test_count,
        "full_test_file_count":    full_test_count,
        "shorter_split":           shorter,
        "longer_split":            "FULL_TEST" if shorter=="TEST" else "TEST",
        "aligned_pair_count":      aligned_count,
        "byte_hash_equal_count":   byte_equal_count,
        "byte_hash_equal_ratio":   round(byte_equal_ratio, 6),
        "first_byte_mismatch_idx": first_mismatch_idx,
        "semantic_result":         semantic_result,
        "relation":                relation,
    }
    audit_rows.append(row)

    # checkpoint 저장
    pd.DataFrame(audit_rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")

# 최종 저장
PREFIX_AUDIT_DF = pd.DataFrame(audit_rows)
prefix_audit_csv = OUTPUT_DIR / "test_fulltest_prefix_audit.csv"
PREFIX_AUDIT_DF.to_csv(prefix_audit_csv, index=False, encoding="utf-8-sig")

print("\n=== TEST/FULL_TEST 관계 요약 ===")
display(PREFIX_AUDIT_DF[["logical_bearing_id","test_file_count","full_test_file_count","relation","byte_hash_equal_ratio"]])

  [1/11] Bearing1_3: TEST=590, FULL_TEST=2375
    [1/11] Bearing1_3: 500/590
  [2/11] Bearing1_4: TEST=1139, FULL_TEST=1428
    [2/11] Bearing1_4: 500/1139
    [2/11] Bearing1_4: 1000/1139
  [3/11] Bearing1_5: TEST=2302, FULL_TEST=2463
    [3/11] Bearing1_5: 500/2302
    [3/11] Bearing1_5: 1000/2302
    [3/11] Bearing1_5: 1500/2302
    [3/11] Bearing1_5: 2000/2302
  [4/11] Bearing1_6: TEST=2232, FULL_TEST=2448
    [4/11] Bearing1_6: 500/2232
    [4/11] Bearing1_6: 1000/2232
    [4/11] Bearing1_6: 1500/2232
    [4/11] Bearing1_6: 2000/2232
  [5/11] Bearing1_7: TEST=1502, FULL_TEST=2259
    [5/11] Bearing1_7: 500/1502
    [5/11] Bearing1_7: 1000/1502
    [5/11] Bearing1_7: 1500/1502
  [6/11] Bearing2_3: TEST=1202, FULL_TEST=1955
    [6/11] Bearing2_3: 500/1202
    [6/11] Bearing2_3: 1000/1202
  [7/11] Bearing2_4: TEST=612, FULL_TEST=751
    [7/11] Bearing2_4: 500/612
  [8/11] Bearing2_5: TEST=2002, FULL_TEST=2311
    [8/11] Bearing2_5: 500/2002
    [8/11] Bearing2_5: 1000/2002
    [8/11]

,logical_bearing_id,test_file_count,full_test_file_count,relation,byte_hash_equal_ratio
0,Bearing1_3,590,2375,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
1,Bearing1_4,1139,1428,DIFFERENT_SEQUENCE,0.000000
2,Bearing1_5,2302,2463,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
3,Bearing1_6,2232,2448,PARTIAL_PREFIX,0.999104
4,Bearing1_7,1502,2259,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
5,Bearing2_3,1202,1955,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
6,Bearing2_4,612,751,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
7,Bearing2_5,2002,2311,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
8,Bearing2_6,572,701,TEST_IS_PREFIX_OF_FULL_TEST,1.000000
9,Bearing2_7,172,230,TEST_IS_PREFIX_OF_FULL_TEST,1.000000


In [8]:
# ============================================================
# 셀 07 — Frozen 및 기존 결과 로드
# ============================================================

import json
import pandas as pd
from pathlib import Path

# BASELINE_M0_frozen.json 로드
frozen_path_candidates = list(PROJECT_ROOT.rglob("BASELINE_M0_frozen.json"))
FROZEN_JSON = None
if frozen_path_candidates:
    FROZEN_PATH = frozen_path_candidates[0]
    with open(FROZEN_PATH, "r", encoding="utf-8") as f:
        FROZEN_JSON = json.load(f)
    print(f"frozen JSON 로드: {FROZEN_PATH}")
    print(json.dumps(FROZEN_JSON, ensure_ascii=False, indent=2))
else:
    print("⚠️  BASELINE_M0_frozen.json 없음 (PARTIAL_AUDIT)")

# m0_baseline_result.csv 로드
result_path_candidates = list(PROJECT_ROOT.rglob("m0_baseline_result.csv"))
LEGACY_DF = None
if result_path_candidates:
    RESULT_PATH = result_path_candidates[0]
    LEGACY_DF = pd.read_csv(RESULT_PATH, encoding="utf-8-sig")
    print(f"\nlegacy 결과 CSV 로드: {RESULT_PATH}")
    display(LEGACY_DF)
else:
    print("⚠️  m0_baseline_result.csv 없음 (PARTIAL_AUDIT)")

# 동일 bearing 이름이 반복될 경우 split 매핑
mapping_rows = []
if LEGACY_DF is not None:
    for _, row in LEGACY_DF.iterrows():
        bid = row.get("bearing", row.get("logical_bearing_id", ""))
        N   = row.get("N", row.get("file_count", None))

        candidates = MANIFEST_DF[MANIFEST_DF["logical_bearing_id"] == bid]
        if len(candidates) == 0:
            status = "NO_MATCH"
            matched_uid = None
        elif len(candidates) == 1:
            status = "UNIQUE_MATCH"
            matched_uid = candidates.iloc[0]["record_uid"]
        else:
            # N으로 disambiguate
            if N is not None:
                n_match = candidates[candidates["file_count"] == int(N)]
                if len(n_match) == 1:
                    status = "UNIQUE_MATCH"
                    matched_uid = n_match.iloc[0]["record_uid"]
                elif len(n_match) == 0:
                    status = "AMBIGUOUS_MATCH"
                    matched_uid = None
                else:
                    status = "AMBIGUOUS_MATCH"
                    matched_uid = None
            else:
                status = "AMBIGUOUS_MATCH"
                matched_uid = None

        mapping_rows.append({
            "legacy_bearing_name": bid,
            "legacy_N":            N,
            "matched_record_uid":  matched_uid,
            "match_status":        status,
        })

    MAPPING_DF = pd.DataFrame(mapping_rows)
    mapping_csv = OUTPUT_DIR / "legacy_result_source_mapping.csv"
    MAPPING_DF.to_csv(mapping_csv, index=False, encoding="utf-8-sig")
    print(f"\n저장: {mapping_csv}")
    display(MAPPING_DF)

frozen JSON 로드: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/BASELINE_M0_frozen.json
{
  "method": "M0_9feat_Mahalanobis",
  "version": "M0_FEMTO_Baseline_v1",
  "date": "2026-07-24",
  "eval_set": "Learning (6 bearings)",
  "axis": "H (COL_H=4, 확정)",
  "params": {
    "K": 6.0,
    "CONSEC": 5,
    "REG_MIN": 90,
    "REG_MAX": 600,
    "FEMTO_FS": 25600,
    "FEMTO_INTERVAL_SEC": 10
  },
  "features": [
    "RMS",
    "Peak",
    "Crest",
    "Kurtosis",
    "Skew",
    "P2P",
    "Std",
    "EnvRMS",
    "HF_ratio"
  ],
  "n_total_learn": 6,
  "n_detected": 6,
  "n_rtf_success": 6,
  "mean_lead_hours": 1.02,
  "max_lead_hours": 3.7,
  "min_lead_hours": 0.1,
  "mean_reg_FAR": 0.0045,
  "max_reg_FAR": 0.0088,
  "pass_criteria": {
    "min_detected": 5,
    "max_reg_FAR": 0.0,
    "note": "M1/M2는 이 기준을 모두 충족해야 우위 인정"
  }
}

legacy 결과 CSV 로드: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/m0_baseline_result.csv


,bearing,N,n_reg,alarm_idx,life_pct,lead_frames,lead_hours,reg_FAR,reg_ok,status
0,Bearing1_3,590,147,NaN,100.0,0,0.0,0.0000,✅,❌ 미탐지
1,Bearing1_4,1139,284,1082.0,95.0,57,0.2,0.0035,⚠️ 0.004,✅ 탐지
2,Bearing1_5,2302,575,NaN,100.0,0,0.0,0.0035,⚠️ 0.003,❌ 미탐지
3,Bearing1_6,2232,558,1630.0,73.0,602,1.7,0.0018,⚠️ 0.002,✅ 탐지
4,Bearing1_7,1502,375,NaN,100.0,0,0.0,0.0027,⚠️ 0.003,❌ 미탐지
5,Bearing2_3,1202,300,NaN,100.0,0,0.0,0.0000,✅,❌ 미탐지
6,Bearing2_4,612,153,269.0,44.0,343,1.0,0.0000,✅,✅ 탐지
7,Bearing2_5,2002,500,NaN,100.0,0,0.0,0.0040,⚠️ 0.004,❌ 미탐지
8,Bearing2_6,572,143,NaN,100.0,0,0.0,0.0070,⚠️ 0.007,❌ 미탐지
9,Bearing2_7,172,90,NaN,100.0,0,0.0,0.0000,✅,❌ 미탐지



저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/legacy_result_source_mapping.csv


,legacy_bearing_name,legacy_N,matched_record_uid,match_status
0,Bearing1_3,590,TEST/Bearing1_3,UNIQUE_MATCH
1,Bearing1_4,1139,TEST/Bearing1_4,UNIQUE_MATCH
2,Bearing1_5,2302,TEST/Bearing1_5,UNIQUE_MATCH
3,Bearing1_6,2232,TEST/Bearing1_6,UNIQUE_MATCH
4,Bearing1_7,1502,TEST/Bearing1_7,UNIQUE_MATCH
5,Bearing2_3,1202,TEST/Bearing2_3,UNIQUE_MATCH
6,Bearing2_4,612,TEST/Bearing2_4,UNIQUE_MATCH
7,Bearing2_5,2002,TEST/Bearing2_5,UNIQUE_MATCH
8,Bearing2_6,572,TEST/Bearing2_6,UNIQUE_MATCH
9,Bearing2_7,172,TEST/Bearing2_7,UNIQUE_MATCH


In [9]:
# ============================================================
# 셀 08 — M0 frozen 기준 구현
# (M0_FEMTO_Baseline_v1_baseline고정.ipynb 기준, 독립 재구현)
# ============================================================

import re
import numpy as np
from pathlib import Path

# ── 등록 길이 공식 (life-length-based bounded registration) ─
# 명칭: "life-length-based bounded registration"
# 한글: "수명 길이 기반 제한형 등록"
# ⚠️ "adaptive registration"이라 부르지 않는다.
def compute_n_reg(N: int) -> int:
    return min(REG_MAX, max(REG_MIN, N // 4))

# ── 신호 로드 ──────────────────────────────────────────────
def load_signal(bearing_dir: Path) -> np.ndarray:
    """
    정렬된 acc_*.csv 파일을 순서대로 읽어
    각 파일에서 COL_H (수평 진동) 값을 추출.
    반환: shape (N,) 1D 배열 (스냅샷 순서)
    """
    files = sorted(
        [f for f in bearing_dir.iterdir()
         if f.name.startswith("acc_") and f.suffix == ".csv"],
        key=lambda f: natural_sort_key(f.name)
    )
    snapshots = []
    for fpath in files:
        try:
            data = np.loadtxt(fpath, delimiter=",")
            if data.ndim == 2 and data.shape[1] > COL_H:
                snapshots.append(data[:, COL_H])
            elif data.ndim == 1:
                snapshots.append(data)
        except Exception as e:
            print(f"    ⚠️  읽기 실패: {fpath.name}: {e}")
    return snapshots  # list of 1D arrays, len = N

# ── 특징 추출 ─────────────────────────────────────────────
def extract_features(snapshot_list: list) -> np.ndarray:
    """
    각 스냅샷(1D 배열)에서 9개 물리 특징 추출.
    반환: shape (N, 9)

    특징 순서:
      0: RMS
      1: Peak
      2: Crest Factor = Peak / RMS
      3: Kurtosis
      4: Skewness
      5: Peak-to-Peak
      6: Std
      7: EnvRMS (절대값의 RMS)
      8: HF_ratio (상위 10% 성분 에너지 비율)
    """
    feats = []
    for sig in snapshot_list:
        sig = sig.astype(np.float64)
        rms     = np.sqrt(np.mean(sig**2))
        peak    = np.max(np.abs(sig))
        crest   = peak / rms if rms > 0 else 0.0
        kurt    = float(np.mean((sig - sig.mean())**4) / (sig.std()**4 + 1e-12))
        skew    = float(np.mean((sig - sig.mean())**3) / (sig.std()**3 + 1e-12))
        p2p     = float(sig.max() - sig.min())
        std     = float(sig.std())
        env_rms = float(np.sqrt(np.mean(np.abs(sig)**2)))
        # HF_ratio: FFT 상위 10% 주파수 대역 에너지 / 전체 에너지
        fft_mag = np.abs(np.fft.rfft(sig))
        total_e = np.sum(fft_mag**2) + 1e-12
        n_hf    = max(1, int(len(fft_mag) * 0.1))
        hf_e    = np.sum(fft_mag[-n_hf:]**2)
        hf_ratio = float(hf_e / total_e)

        feats.append([rms, peak, crest, kurt, skew, p2p, std, env_rms, hf_ratio])
    return np.array(feats)  # (N, 9)

# ── Mahalanobis 거리 ──────────────────────────────────────
def mahal(X: np.ndarray, mean: np.ndarray, cov_inv: np.ndarray) -> np.ndarray:
    diff = X - mean
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))

# ── 연속 경보 판정 ─────────────────────────────────────────
def consecutive_alarm(exceed: np.ndarray, consec: int) -> dict:
    """
    exceed: boolean array (True = 임계 초과)
    consec: 연속 조건 수

    반환:
      alarm_start_idx   : CONSEC 연속 초과가 시작된 첫 번째 index
      alarm_confirm_idx : 실제로 CONSEC 조건이 충족된 마지막 index
                         (= alarm_start_idx + CONSEC - 1)
      alarm_events      : alarm event 수
    """
    alarm_start_idx   = None
    alarm_confirm_idx = None
    alarm_events      = 0
    in_alarm          = False
    run               = 0
    run_start         = None
    max_consec_run    = 0
    cur_run           = 0

    for i, v in enumerate(exceed):
        if v:
            if run == 0:
                run_start = i
            run    += 1
            cur_run = run
            if run == consec and not in_alarm:
                alarm_start_idx   = run_start
                alarm_confirm_idx = i
                alarm_events     += 1
                in_alarm          = True
        else:
            max_consec_run = max(max_consec_run, cur_run)
            run     = 0
            cur_run = 0
            in_alarm = False

    max_consec_run = max(max_consec_run, cur_run)

    return {
        "alarm_start_idx":   alarm_start_idx,
        "alarm_confirm_idx": alarm_confirm_idx,
        "alarm_events":      alarm_events,
        "max_consec_run":    max_consec_run,
    }

# ── bearing 한 개 전체 실행 ──────────────────────────────
def run_one(bearing_dir: Path, record_uid: str) -> dict:
    """
    단일 bearing에 대해 M0 frozen 기준 구현 전체 실행.
    bearing 한 개씩 처리, 메모리에 전체 waveform 적재 금지.
    """
    snapshot_list = load_signal(bearing_dir)
    N = len(snapshot_list)
    if N == 0:
        return {"record_uid": record_uid, "error": "NO_DATA"}

    n_reg     = compute_n_reg(N)
    feat_mat  = extract_features(snapshot_list)  # (N, 9)

    reg_feat  = feat_mat[:n_reg]
    mean_vec  = reg_feat.mean(axis=0)
    cov_mat   = np.cov(reg_feat.T)
    # 정규화 (공분산 안정화)
    cov_reg   = cov_mat + np.eye(cov_mat.shape[0]) * 1e-6
    cov_inv   = np.linalg.inv(cov_reg)

    threshold = K  # Mahalanobis 거리 임계값 = K (단위: std)

    # 전체 구간 Mahalanobis 거리
    distances = mahal(feat_mat, mean_vec, cov_inv)

    exceed    = distances > threshold
    alarm_res = consecutive_alarm(exceed, CONSEC)

    alarm_start   = alarm_res["alarm_start_idx"]
    alarm_confirm = alarm_res["alarm_confirm_idx"]

    # 등록 구간 진단 지표
    reg_exceed   = exceed[:n_reg]
    reg_FAR      = float(np.mean(reg_exceed))
    reg_alarm_r  = consecutive_alarm(reg_exceed, CONSEC)
    reg_max_consec = reg_alarm_r["max_consec_run"]
    reg_event_cnt  = reg_alarm_r["alarm_events"]

    # alarm_state_window: alarm_start_idx 이후 CONSEC이 충족된 상태인 윈도우 수
    # (단순화: exceed 배열에서 연속 alarm 구간의 윈도우 총합)
    in_state = False
    reg_alarm_state_windows = 0
    run = 0
    for v in reg_exceed:
        if v:
            run += 1
            if run >= CONSEC:
                in_state = True
        else:
            run = 0
            in_state = False
        if in_state:
            reg_alarm_state_windows += 1

    reg_alarm_state_window_rate = (
        reg_alarm_state_windows / n_reg if n_reg > 0 else 0.0
    )

    # Held-out 진단 (fit: 앞 70%, eval: 뒤 30%)
    n_fit  = int(n_reg * 0.7)
    n_eval = n_reg - n_fit
    fit_feat  = feat_mat[:n_fit]
    eval_feat = feat_mat[n_fit:n_reg]
    heldout_far = None
    if n_fit > 1 and n_eval > 0:
        mean_fit = fit_feat.mean(axis=0)
        cov_fit  = np.cov(fit_feat.T) + np.eye(9) * 1e-6
        cov_fit_inv = np.linalg.inv(cov_fit)
        dist_eval = mahal(eval_feat, mean_fit, cov_fit_inv)
        heldout_far = float(np.mean(dist_eval > threshold))

    # 리드타임 계산
    if alarm_start is not None:
        life_pct_start   = alarm_start / N * 100
        life_pct_confirm = alarm_confirm / N * 100
        lead_frames_start   = N - alarm_start
        lead_frames_confirm = N - alarm_confirm
        lead_hours_start    = lead_frames_start  * FEMTO_DECISION_INTERVAL_SEC / 3600
        lead_hours_confirm  = lead_frames_confirm * FEMTO_DECISION_INTERVAL_SEC / 3600
    else:
        life_pct_start = life_pct_confirm = None
        lead_frames_start = lead_frames_confirm = None
        lead_hours_start  = lead_hours_confirm  = None

    return {
        "record_uid":                   record_uid,
        "N":                            N,
        "n_reg":                        n_reg,
        "threshold":                    threshold,
        "alarm_start_idx":              alarm_start,
        "alarm_confirm_idx":            alarm_confirm,
        "life_pct_start":               life_pct_start,
        "life_pct_confirm":             life_pct_confirm,
        "lead_frames_start":            lead_frames_start,
        "lead_frames_confirm":          lead_frames_confirm,
        "lead_hours_start":             lead_hours_start,
        "lead_hours_confirm":           lead_hours_confirm,
        "reg_window_exceedance_rate":   reg_FAR,
        "reg_max_consecutive_exceedances": reg_max_consec,
        "reg_alarm_event_count":        reg_event_cnt,
        "reg_alarm_state_window_count": reg_alarm_state_windows,
        "reg_alarm_state_window_rate":  reg_alarm_state_window_rate,
        "diagnostic_heldout_reg_exceedance_rate": heldout_far,
    }

print("[M0 기준 구현] 함수 정의 완료 ✅")
print("  등록 방식: life-length-based bounded registration (수명 길이 기반 제한형 등록)")
print("  n_reg = min(REG_MAX, max(REG_MIN, N // 4))")

[M0 기준 구현] 함수 정의 완료 ✅
  등록 방식: life-length-based bounded registration (수명 길이 기반 제한형 등록)
  n_reg = min(REG_MAX, max(REG_MIN, N // 4))


In [10]:
# ============================================================
# 셀 09 — Learning 6개 M0 재계산
# ============================================================

import pandas as pd

LEARNING_TARGETS = [
    "Bearing1_1", "Bearing1_2",
    "Bearing2_1", "Bearing2_2",
    "Bearing3_1", "Bearing3_2",
]

recomputed_rows = []

if RECOMPUTE_LEARNING_M0:
    for i, bid in enumerate(LEARNING_TARGETS):
        bearing_dir = LEARNING_DIR / bid
        record_uid  = f"LEARNING/{bid}"
        print(f"  [{i+1}/{len(LEARNING_TARGETS)}] {record_uid} 재계산 중...")
        result = run_one(bearing_dir, record_uid)
        recomputed_rows.append(result)
        print(f"    N={result.get('N')}, n_reg={result.get('n_reg')}, "
              f"alarm_start={result.get('alarm_start_idx')}, "
              f"alarm_confirm={result.get('alarm_confirm_idx')}, "
              f"reg_FAR={result.get('reg_window_exceedance_rate'):.4f}")
else:
    print("RECOMPUTE_LEARNING_M0=False — 재계산 건너뜀")

RECOMPUTED_DF = pd.DataFrame(recomputed_rows)
recomputed_csv = OUTPUT_DIR / "m0_learning_recomputed.csv"
RECOMPUTED_DF.to_csv(recomputed_csv, index=False, encoding="utf-8-sig")

print(f"\n저장: {recomputed_csv}")
display(RECOMPUTED_DF[[
    "record_uid", "N", "n_reg",
    "alarm_start_idx", "alarm_confirm_idx",
    "life_pct_start", "lead_hours_start",
    "reg_window_exceedance_rate"
]])

  [1/6] LEARNING/Bearing1_1 재계산 중...
    N=2803, n_reg=600, alarm_start=1414, alarm_confirm=1418, reg_FAR=0.0250
  [2/6] LEARNING/Bearing1_2 재계산 중...
    N=871, n_reg=217, alarm_start=716, alarm_confirm=720, reg_FAR=0.0369
  [3/6] LEARNING/Bearing2_1 재계산 중...
    N=911, n_reg=227, alarm_start=874, alarm_confirm=878, reg_FAR=0.0132
  [4/6] LEARNING/Bearing2_2 재계산 중...
    N=797, n_reg=199, alarm_start=204, alarm_confirm=208, reg_FAR=0.0151
  [5/6] LEARNING/Bearing3_1 재계산 중...
    N=515, n_reg=128, alarm_start=479, alarm_confirm=483, reg_FAR=0.0234
  [6/6] LEARNING/Bearing3_2 재계산 중...
    N=1637, n_reg=409, alarm_start=1422, alarm_confirm=1426, reg_FAR=0.0147

저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/m0_learning_recomputed.csv


,record_uid,N,n_reg,alarm_start_idx,alarm_confirm_idx,life_pct_start,lead_hours_start,reg_window_exceedance_rate
0,LEARNING/Bearing1_1,2803,600,1414,1418,50.445951,3.858333,0.025000
1,LEARNING/Bearing1_2,871,217,716,720,82.204363,0.430556,0.036866
2,LEARNING/Bearing2_1,911,227,874,878,95.938529,0.102778,0.013216
3,LEARNING/Bearing2_2,797,199,204,208,25.595985,1.647222,0.015075
4,LEARNING/Bearing3_1,515,128,479,483,93.009709,0.100000,0.023438
5,LEARNING/Bearing3_2,1637,409,1422,1426,86.866219,0.597222,0.014670


In [11]:
# ============================================================
# 셀 10 — reg_FAR 의미 감사
# ============================================================

import pandas as pd
import numpy as np

print("="*60)
print("reg_FAR (reg_window_exceedance_rate) 의미 감사")
print("="*60)
print()
print("[정확한 수식]")
print("  reg_window_exceedance_rate = np.mean(distances[:n_reg] > threshold)")
print()
print("[의미 해석]")
print("  유형: FAR-A")
print("  학습/평가 관계: IN_SAMPLE")
print("  이유: 동일 registration 구간으로 mean·공분산·threshold를 생성하고")
print("        동일 registration 구간에서 exceedance rate를 계산한다.")
print()
print("  ⚠️  기존 reg_FAR는 등록 구간 단일 윈도우 임계값 초과율이다.")
print("      CONSEC=5 적용 후 alarm event rate가 아니다.")

if len(recomputed_rows) > 0:
    far_df = pd.DataFrame([
        {
            "record_uid":             r["record_uid"],
            "n_reg":                  r["n_reg"],
            "reg_FAR_type":           "FAR-A",
            "reg_sample_relation":    "IN_SAMPLE",
            "reg_window_exceedance_rate":         r["reg_window_exceedance_rate"],
            "reg_max_consecutive_exceedances":    r["reg_max_consecutive_exceedances"],
            "reg_alarm_event_count":              r["reg_alarm_event_count"],
            "reg_alarm_state_window_rate":        r["reg_alarm_state_window_rate"],
            "diagnostic_heldout_reg_exceedance_rate": r.get("diagnostic_heldout_reg_exceedance_rate"),
        }
        for r in recomputed_rows
    ])
    far_csv = OUTPUT_DIR / "reg_far_semantics.csv"
    far_df.to_csv(far_csv, index=False, encoding="utf-8-sig")
    print(f"\n저장: {far_csv}")
    display(far_df)

    print()
    print("[Held-out 진단 주의]")
    print("  - fit: 등록 앞 70%, eval: 등록 뒤 30%")
    print("  - frozen 성적에 사용하지 않음")
    print("  - pass/fail에 사용하지 않음")
    print("  - 과적합/낙관 편향 확인용 참고값")

reg_FAR (reg_window_exceedance_rate) 의미 감사

[정확한 수식]
  reg_window_exceedance_rate = np.mean(distances[:n_reg] > threshold)

[의미 해석]
  유형: FAR-A
  학습/평가 관계: IN_SAMPLE
  이유: 동일 registration 구간으로 mean·공분산·threshold를 생성하고
        동일 registration 구간에서 exceedance rate를 계산한다.

  ⚠️  기존 reg_FAR는 등록 구간 단일 윈도우 임계값 초과율이다.
      CONSEC=5 적용 후 alarm event rate가 아니다.

저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/reg_far_semantics.csv


,record_uid,n_reg,reg_FAR_type,reg_sample_relation,reg_window_exceedance_rate,reg_max_consecutive_exceedances,reg_alarm_event_count,reg_alarm_state_window_rate,diagnostic_heldout_reg_exceedance_rate
0,LEARNING/Bearing1_1,600,FAR-A,IN_SAMPLE,0.025000,2,0,0.0,0.066667
1,LEARNING/Bearing1_2,217,FAR-A,IN_SAMPLE,0.036866,2,0,0.0,0.090909
2,LEARNING/Bearing2_1,227,FAR-A,IN_SAMPLE,0.013216,2,0,0.0,0.086957
3,LEARNING/Bearing2_2,199,FAR-A,IN_SAMPLE,0.015075,1,0,0.0,0.500000
4,LEARNING/Bearing3_1,128,FAR-A,IN_SAMPLE,0.023438,1,0,0.0,0.102564
5,LEARNING/Bearing3_2,409,FAR-A,IN_SAMPLE,0.014670,1,0,0.0,0.008130



[Held-out 진단 주의]
  - fit: 등록 앞 70%, eval: 등록 뒤 30%
  - frozen 성적에 사용하지 않음
  - pass/fail에 사용하지 않음
  - 과적합/낙관 편향 확인용 참고값


In [12]:
# ============================================================
# 셀 11 — 경보 시간 감사
# ============================================================

print("="*60)
print("경보 시간 감사")
print("="*60)
print()
print(f"  snapshot_duration_sec  = {FEMTO_SNAPSHOT_DURATION_SEC} s  (스냅샷 1개 길이)")
print(f"  decision_interval_sec  = {FEMTO_DECISION_INTERVAL_SEC} s  (의사결정 파일 간격)")
print()
print(f"  CONSEC = {CONSEC}에 대한 경보 시간 의미:")
print(f"    decision span            = {CONSEC} snapshots")
print(f"    nominal accumulated intervals = {(CONSEC-1)*FEMTO_DECISION_INTERVAL_SEC:.0f} seconds")
print(f"    first exceedance to confirmation timestamp diff = {(CONSEC-1)*FEMTO_DECISION_INTERVAL_SEC:.0f} seconds")
print()
print("  alarm_start_idx  : CONSEC개의 연속 초과가 시작된 첫 번째 index")
print("  alarm_confirm_idx: 실제로 CONSEC 조건이 충족된 마지막 index")
print("  alarm_confirm_idx = alarm_start_idx + CONSEC - 1")
print()
print("  [frozen 결과 비교 기준]")
print("    - legacy_alarm_start_idx 기준으로 frozen 결과와 비교")
print("    - 실제 현장 경보 시점 설명에는 alarm_confirm_idx 기준 병기")

# Learning 재계산 결과 표시
if len(recomputed_rows) > 0:
    import pandas as pd
    timing_df = pd.DataFrame([
        {
            "record_uid":            r["record_uid"],
            "N":                     r["N"],
            "legacy_alarm_start_idx":  r["alarm_start_idx"],
            "alarm_confirm_idx":      r["alarm_confirm_idx"],
            "life_pct_start":         r["life_pct_start"],
            "life_pct_confirm":       r["life_pct_confirm"],
            "lead_hours_start":       r["lead_hours_start"],
            "lead_hours_confirm":     r["lead_hours_confirm"],
        }
        for r in recomputed_rows
    ])
    print()
    display(timing_df)

경보 시간 감사

  snapshot_duration_sec  = 0.1 s  (스냅샷 1개 길이)
  decision_interval_sec  = 10.0 s  (의사결정 파일 간격)

  CONSEC = 5에 대한 경보 시간 의미:
    decision span            = 5 snapshots
    nominal accumulated intervals = 40 seconds
    first exceedance to confirmation timestamp diff = 40 seconds

  alarm_start_idx  : CONSEC개의 연속 초과가 시작된 첫 번째 index
  alarm_confirm_idx: 실제로 CONSEC 조건이 충족된 마지막 index
  alarm_confirm_idx = alarm_start_idx + CONSEC - 1

  [frozen 결과 비교 기준]
    - legacy_alarm_start_idx 기준으로 frozen 결과와 비교
    - 실제 현장 경보 시점 설명에는 alarm_confirm_idx 기준 병기



,record_uid,N,legacy_alarm_start_idx,alarm_confirm_idx,life_pct_start,life_pct_confirm,lead_hours_start,lead_hours_confirm
0,LEARNING/Bearing1_1,2803,1414,1418,50.445951,50.588655,3.858333,3.847222
1,LEARNING/Bearing1_2,871,716,720,82.204363,82.663605,0.430556,0.419444
2,LEARNING/Bearing2_1,911,874,878,95.938529,96.377607,0.102778,0.091667
3,LEARNING/Bearing2_2,797,204,208,25.595985,26.097867,1.647222,1.636111
4,LEARNING/Bearing3_1,515,479,483,93.009709,93.786408,0.100000,0.088889
5,LEARNING/Bearing3_2,1637,1422,1426,86.866219,87.110568,0.597222,0.586111


In [13]:
# ============================================================
# 셀 12 — Frozen 일관성 검사
# ============================================================

import json
import pandas as pd
import numpy as np

def compare_values(frozen_val, recomputed_val, abs_tol=ABS_TOL, rounding_tol=ROUNDING_TOL):
    """exact_match / rounded_match 비교."""
    if frozen_val is None or recomputed_val is None:
        return {"exact_match": None, "rounded_match": None}
    try:
        exact   = abs(float(frozen_val) - float(recomputed_val)) < abs_tol
        rounded = abs(float(frozen_val) - float(recomputed_val)) < rounding_tol
        return {"exact_match": exact, "rounded_match": rounded}
    except Exception:
        return {"exact_match": str(frozen_val)==str(recomputed_val), "rounded_match": None}

print("="*60)
print("Frozen 일관성 검사")
print("="*60)

consistency_rows = []

if FROZEN_JSON is None:
    print("⚠️  FROZEN_JSON 없음 — 일관성 검사 건너뜀")
elif len(recomputed_rows) == 0:
    print("⚠️  재계산 결과 없음 — 일관성 검사 건너뜀")
else:
    rec_df = RECOMPUTED_DF.copy()

    # n_detected: alarm_start_idx가 None이 아닌 건수
    n_detected = int(rec_df["alarm_start_idx"].notna().sum())
    n_total    = len(rec_df)

    detected_rows = rec_df[rec_df["alarm_start_idx"].notna()]
    mean_lead = float(detected_rows["lead_hours_start"].mean()) if len(detected_rows) > 0 else None
    max_lead  = float(detected_rows["lead_hours_start"].max())  if len(detected_rows) > 0 else None
    min_lead  = float(detected_rows["lead_hours_start"].min())  if len(detected_rows) > 0 else None
    mean_far  = float(rec_df["reg_window_exceedance_rate"].mean())
    max_far   = float(rec_df["reg_window_exceedance_rate"].max())

    check_fields = [
        ("n_total_learn",  n_total,    FROZEN_JSON.get("n_total_learn")),
        ("n_detected",     n_detected, FROZEN_JSON.get("n_detected")),
        ("mean_lead_hours",mean_lead,  FROZEN_JSON.get("mean_lead_hours")),
        ("max_lead_hours", max_lead,   FROZEN_JSON.get("max_lead_hours")),
        ("min_lead_hours", min_lead,   FROZEN_JSON.get("min_lead_hours")),
        ("mean_reg_FAR",   mean_far,   FROZEN_JSON.get("mean_reg_FAR")),
        ("max_reg_FAR",    max_far,    FROZEN_JSON.get("max_reg_FAR")),
    ]

    for field, recomp_val, frozen_val in check_fields:
        cmp = compare_values(frozen_val, recomp_val)
        consistency_rows.append({
            "field":          field,
            "frozen_value":   frozen_val,
            "recomputed_value": recomp_val,
            "exact_match":    cmp["exact_match"],
            "rounded_match":  cmp["rounded_match"],
        })

    CONSISTENCY_DF = pd.DataFrame(consistency_rows)
    consist_csv    = OUTPUT_DIR / "frozen_consistency_report.csv"
    CONSISTENCY_DF.to_csv(consist_csv, index=False, encoding="utf-8-sig")

    print(f"저장: {consist_csv}")
    display(CONSISTENCY_DF)

    # Pass criteria 일관성
    print("\n[ORIGINAL_FROZEN_CRITERIA]")
    orig_min_detected = FROZEN_JSON.get("min_detected", 5)
    orig_max_reg_far  = FROZEN_JSON.get("max_reg_FAR",  0.0)
    print(f"  min_detected ≥ {orig_min_detected}")
    print(f"  max_reg_FAR  ≤ {orig_max_reg_far}")

    detect_pass = n_detected >= orig_min_detected
    far_pass    = max_far    <= orig_max_reg_far
    overall     = detect_pass and far_pass

    print(f"\n  detection_criterion_pass  = {detect_pass}  (n_detected={n_detected})")
    print(f"  reg_far_criterion_pass    = {far_pass}  (max_reg_FAR={max_far:.6f})")
    print(f"  overall_original_criterion_pass = {overall}")

    print("\n[PROPOSED_CRITERIA_NOT_APPLIED]")
    print("  새로운 기준이 제안되더라도 이번 감사에서 자동 반영하지 않음")
    print("  pass criteria는 frozen JSON에 명시된 원래 기준만 적용")

Frozen 일관성 검사
저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/frozen_consistency_report.csv


,field,frozen_value,recomputed_value,exact_match,rounded_match
0,n_total_learn,6.0000,6.000000,True,True
1,n_detected,6.0000,6.000000,True,True
2,mean_lead_hours,1.0200,1.122685,False,False
3,max_lead_hours,3.7000,3.858333,False,False
4,min_lead_hours,0.1000,0.100000,True,True
5,mean_reg_FAR,0.0045,0.021378,False,False
6,max_reg_FAR,0.0088,0.036866,False,False



[ORIGINAL_FROZEN_CRITERIA]
  min_detected ≥ 5
  max_reg_FAR  ≤ 0.0088

  detection_criterion_pass  = True  (n_detected=6)
  reg_far_criterion_pass    = False  (max_reg_FAR=0.036866)
  overall_original_criterion_pass = False

[PROPOSED_CRITERIA_NOT_APPLIED]
  새로운 기준이 제안되더라도 이번 감사에서 자동 반영하지 않음
  pass criteria는 frozen JSON에 명시된 원래 기준만 적용


In [14]:
# ============================================================
# 셀 13 — 원본 불변성 재검사
# ============================================================

import json
import pandas as pd

print("=== 원본 파일 사후 스냅샷 (AFTER) ===")
SOURCE_SNAP_AFTER = snap_source_files("after")

# 저장
after_path = OUTPUT_DIR / "source_file_snapshot_after.json"
with open(after_path, "w", encoding="utf-8") as f:
    json.dump(SOURCE_SNAP_AFTER, f, ensure_ascii=False, indent=2)

# 비교
immut_rows = []
SOURCE_FILES_UNCHANGED = True

for fname in SOURCE_FILE_NAMES:
    before = SOURCE_SNAP_BEFORE.get(fname, {})
    after  = SOURCE_SNAP_AFTER.get(fname,  {})

    if before.get("status") == "MISSING" and after.get("status") == "MISSING":
        unchanged = True  # 처음부터 없었으면 불변
    elif before.get("status") == "MISSING" or after.get("status") == "MISSING":
        unchanged = False
        SOURCE_FILES_UNCHANGED = False
    else:
        h_before = before.get("sha256_before", "")
        h_after  = after.get("sha256_after",   "")
        unchanged = (h_before == h_after)
        if not unchanged:
            SOURCE_FILES_UNCHANGED = False

    immut_rows.append({
        "filename":      fname,
        "sha256_before": before.get("sha256_before", "MISSING"),
        "sha256_after":  after.get("sha256_after",   "MISSING"),
        "unchanged":     unchanged,
    })

IMMUT_DF = pd.DataFrame(immut_rows)
immut_csv = OUTPUT_DIR / "source_file_immutability_check.csv"
IMMUT_DF.to_csv(immut_csv, index=False, encoding="utf-8-sig")

print(f"\nsource_files_unchanged = {SOURCE_FILES_UNCHANGED}")
if not SOURCE_FILES_UNCHANGED:
    print("🚨 AUDIT_INVALID_SOURCE_MODIFIED: 원본 파일이 변경됨!")
else:
    print("✅ 원본 파일 변경 없음")

print(f"저장: {immut_csv}")
display(IMMUT_DF)

=== 원본 파일 사후 스냅샷 (AFTER) ===
  [after] BASELINE_M0_frozen.json: 783 bytes  sha256=a4632ad1b9c51976...
  [after] m0_baseline_result.csv: 1,899 bytes  sha256=63f493e6e3482157...
  [after] M0_FEMTO_Baseline_v1_baseline고정.ipynb: MISSING

source_files_unchanged = True
✅ 원본 파일 변경 없음
저장: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125/source_file_immutability_check.csv


,filename,sha256_before,sha256_after,unchanged
0,BASELINE_M0_frozen.json,a4632ad1b9c519768081cea2c35238598f92144580573d...,a4632ad1b9c519768081cea2c35238598f92144580573d...,True
1,m0_baseline_result.csv,63f493e6e3482157dba7a3b7eef75d6d679086bc2ae5b2...,63f493e6e3482157dba7a3b7eef75d6d679086bc2ae5b2...,True
2,M0_FEMTO_Baseline_v1_baseline고정.ipynb,MISSING,MISSING,True


In [15]:
# ============================================================
# 셀 14 — 최종 보고서 생성
# ============================================================

import json
from datetime import datetime

# ── Audit status 결정 ─────────────────────────────────────
if not SOURCE_FILES_UNCHANGED:
    AUDIT_STATUS = "AUDIT_INVALID_SOURCE_MODIFIED"
elif AUDIT_COMPLETENESS == "PARTIAL_AUDIT_MISSING_REFERENCE":
    AUDIT_STATUS = "AUDIT_PARTIAL"
elif len(recomputed_rows) == 0:
    AUDIT_STATUS = "AUDIT_PARTIAL"
else:
    # 일관성 검사 결과 활용
    warnings = []
    if FROZEN_JSON is not None and len(consistency_rows) > 0:
        all_rounded = all(
            r.get("rounded_match") in (True, None)
            for r in consistency_rows
        )
        all_exact = all(
            r.get("exact_match") in (True, None)
            for r in consistency_rows
        )
        if not all_rounded:
            AUDIT_STATUS = "AUDIT_FAILED"
        elif all_exact:
            AUDIT_STATUS = "AUDIT_PASS"
        else:
            AUDIT_STATUS = "AUDIT_PASS_WITH_WARNINGS"
    else:
        AUDIT_STATUS = "AUDIT_PARTIAL"

# ── Recommended next action ─────────────────────────────
if AUDIT_STATUS == "AUDIT_PASS":
    NEXT_ACTION = "READY_FOR_M0_BRIDGE_2048"
elif AUDIT_STATUS == "AUDIT_PASS_WITH_WARNINGS":
    NEXT_ACTION = "MANUAL_REVIEW_REQUIRED"
elif AUDIT_STATUS == "AUDIT_INVALID_SOURCE_MODIFIED":
    NEXT_ACTION = "FIX_DATA_IDENTITY_FIRST"
elif AUDIT_STATUS == "AUDIT_PARTIAL":
    NEXT_ACTION = "FIX_REFERENCE_MAPPING_FIRST"
else:
    NEXT_ACTION = "RECOMPUTE_M0_REQUIRED"

# ── Markdown 보고서 ──────────────────────────────────────
ts_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

md_lines = [
    f"# FINAL AUDIT SUMMARY — {AUDIT_VERSION}",
    f"",
    f"생성 시각: {ts_now}",
    f"",
    f"---",
    f"",
    f"## 1. Audit Status",
    f"",
    f"```",
    f"{AUDIT_STATUS}",
    f"```",
    f"",
    f"## 2. Dataset Structure",
    f"",
    f"| Split | 실제 수 | 기대 수 | 일치 |",
    f"|:---|---:|---:|---:|",
    f"| LEARNING  | {actual_learning}  | {EXPECTED_LEARNING_RECORDS}  | {actual_learning==EXPECTED_LEARNING_RECORDS} |",
    f"| TEST      | {actual_test}  | {EXPECTED_TEST_RECORDS}  | {actual_test==EXPECTED_TEST_RECORDS} |",
    f"| FULL_TEST | {actual_full_test}  | {EXPECTED_FULL_TEST_RECORDS}  | {actual_full_test==EXPECTED_FULL_TEST_RECORDS} |",
    f"| **TOTAL** | **{actual_total}** | **{EXPECTED_TOTAL_RECORDS}** | **{actual_total==EXPECTED_TOTAL_RECORDS}** |",
    f"",
    f"## 3. Duplicate-Name Root Cause",
    f"",
    f"기존 M0 코드는 전체 경로가 아니라 `bd.name`을 결과 ID로 사용했기 때문에",
    f"`TEST/BearingX_Y`와 `FULL_TEST/BearingX_Y`가 동일 이름으로 표시됐다.",
    f"이는 데이터 삭제나 병합이 아니며, basename 사용에 따른 표기상 중복이다.",
    f"",
    f"## 4. TEST/FULL_TEST Relation",
    f"",
]

if 'PREFIX_AUDIT_DF' in dir() and len(PREFIX_AUDIT_DF) > 0:
    md_lines += [
        f"| bearing | Test N | Full Test N | relation | hash ratio | evidence |",
        f"|:---|---:|---:|:---|---:|:---|",
    ]
    for _, row in PREFIX_AUDIT_DF.iterrows():
        md_lines.append(
            f"| {row['logical_bearing_id']} "
            f"| {row.get('test_file_count', '?')} "
            f"| {row.get('full_test_file_count', '?')} "
            f"| {row.get('relation', '?')} "
            f"| {row.get('byte_hash_equal_ratio', '?')} "
            f"| {row.get('semantic_result', 'N/A')} |"
        )
else:
    md_lines.append("(감사 미완료)")

md_lines += [
    f"",
    f"## 5. M0 Learning Reproduction",
    f"",
]

if len(recomputed_rows) > 0 and FROZEN_JSON is not None:
    md_lines += [
        f"| record_uid | legacy alarm | recomputed alarm | legacy FAR | recomputed FAR | match |",
        f"|:---|---:|---:|---:|---:|:---|",
    ]
    for r in recomputed_rows:
        bid = r["record_uid"].split("/")[-1]
        legacy_alarm = "N/A"
        legacy_far   = "N/A"
        match_str    = "N/A"
        if LEGACY_DF is not None:
            leg = LEGACY_DF[LEGACY_DF.iloc[:,0].astype(str).str.contains(bid)]
            if len(leg) > 0:
                legacy_alarm = leg.iloc[0].get("alarm_idx", "N/A")
                legacy_far   = leg.iloc[0].get("reg_FAR",   "N/A")
        recomp_alarm = r.get("alarm_start_idx", "N/A")
        recomp_far   = round(r.get("reg_window_exceedance_rate", 0), 6)
        md_lines.append(
            f"| {r['record_uid']} | {legacy_alarm} | {recomp_alarm} | {legacy_far} | {recomp_far} | {match_str} |"
        )

md_lines += [
    f"",
    f"## 6. reg_FAR Definition",
    f"",
    f"- **유형**: FAR-A",
    f"- **학습/평가 관계**: IN_SAMPLE",
    f"- **수식**: `reg_window_exceedance_rate = np.mean(distances[:n_reg] > threshold)`",
    f"- **의미**: 등록 구간 단일 윈도우 임계값 초과율 (CONSEC=5 alarm event rate 아님)",
    f"",
    f"## 7. Alarm Timing",
    f"",
    f"- `snapshot_duration_sec` = {FEMTO_SNAPSHOT_DURATION_SEC} s (스냅샷 길이)",
    f"- `decision_interval_sec` = {FEMTO_DECISION_INTERVAL_SEC} s (의사결정 파일 간격)",
    f"- `alarm_start_idx`   : CONSEC 연속 초과가 시작된 첫 번째 index",
    f"- `alarm_confirm_idx` : 실제로 CONSEC 조건이 충족된 마지막 index",
    f"- `alarm_confirm_idx = alarm_start_idx + CONSEC - 1`",
    f"- frozen 비교: legacy_alarm_start_idx 기준 사용",
    f"- 현장 경보 설명: alarm_confirm_idx 기준 병기",
    f"",
    f"## 8. Frozen Consistency",
    f"",
]

if len(consistency_rows) > 0:
    md_lines += [
        f"| field | frozen | recomputed | exact_match | rounded_match |",
        f"|:---|---:|---:|:---:|:---:|",
    ]
    for r in consistency_rows:
        md_lines.append(
            f"| {r['field']} | {r['frozen_value']} | {round(r['recomputed_value'],4) if r['recomputed_value'] is not None else 'N/A'} | {r['exact_match']} | {r['rounded_match']} |"
        )
else:
    md_lines.append("(frozen JSON 없음)")

md_lines += [
    f"",
    f"## 9. Original Pass Criteria",
    f"",
    f"**ORIGINAL_FROZEN_CRITERIA** (frozen JSON 기준):",
]

if FROZEN_JSON:
    md_lines += [
        f"- `min_detected` ≥ {FROZEN_JSON.get('min_detected', 'N/A')}",
        f"- `max_reg_FAR`  ≤ {FROZEN_JSON.get('max_reg_FAR',  'N/A')}",
    ]

md_lines += [
    f"",
    f"**PROPOSED_CRITERIA_NOT_APPLIED**: 새 기준은 이번 감사에 반영하지 않음.",
    f"",
    f"## 10. Source Immutability",
    f"",
    f"- `source_files_unchanged` = {SOURCE_FILES_UNCHANGED}",
]

for r in immut_rows:
    icon = "✅" if r["unchanged"] else "🚨"
    md_lines.append(f"  - {icon} `{r['filename']}` : {r['sha256_before'][:12]}... → {r['sha256_after'][:12]}...")

md_lines += [
    f"",
    f"## 11. Recommended Next Action",
    f"",
    f"```",
    f"{NEXT_ACTION}",
    f"```",
]

md_text = "\n".join(md_lines)

# JSON 보고서
audit_json = {
    "audit_version":                AUDIT_VERSION,
    "generated_at":                 ts_now,
    "audit_status":                 AUDIT_STATUS,
    "dataset_structure": {
        "learning":   actual_learning,
        "test":       actual_test,
        "full_test":  actual_full_test,
        "total":      actual_total,
        "expected_match": manifest_match,
    },
    "duplicate_name_root_cause":    "bd.name 사용으로 TEST/FULL_TEST 동일 이름 표시",
    "source_files_unchanged":       SOURCE_FILES_UNCHANGED,
    "reg_FAR_type":                 "FAR-A",
    "reg_FAR_relation":             "IN_SAMPLE",
    "alarm_start_is_legacy_basis":  True,
    "alarm_confirm_idx_formula":    "alarm_start_idx + CONSEC - 1",
    "recommended_next_action":      NEXT_ACTION,
    "consistency_rows":             consistency_rows,
    "original_pass_criteria":       {
        "min_detected": FROZEN_JSON.get("min_detected") if FROZEN_JSON else None,
        "max_reg_FAR":  FROZEN_JSON.get("max_reg_FAR")  if FROZEN_JSON else None,
    },
}

# 저장
md_path   = OUTPUT_DIR / "FINAL_AUDIT_SUMMARY.md"
json_path = OUTPUT_DIR / "FINAL_AUDIT_SUMMARY.json"

with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_text)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(audit_json, f, ensure_ascii=False, indent=2)

# audit_config.json 저장
config_out = {
    "AUDIT_VERSION":               AUDIT_VERSION,
    "FEMTO_FS":                    FEMTO_FS,
    "FEMTO_SNAPSHOT_DURATION_SEC": FEMTO_SNAPSHOT_DURATION_SEC,
    "FEMTO_DECISION_INTERVAL_SEC": FEMTO_DECISION_INTERVAL_SEC,
    "COL_H":                       COL_H,
    "K":                           K,
    "CONSEC":                      CONSEC,
    "REG_MIN":                     REG_MIN,
    "REG_MAX":                     REG_MAX,
    "FULL_BYTE_HASH":              FULL_BYTE_HASH,
    "SEMANTIC_HASH_ON_BYTE_MISMATCH": SEMANTIC_HASH_ON_BYTE_MISMATCH,
    "RECOMPUTE_LEARNING_M0":       RECOMPUTE_LEARNING_M0,
    "RECOMPUTE_NON_LEARNING_M0":   RECOMPUTE_NON_LEARNING_M0,
}
config_path = OUTPUT_DIR / "audit_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_out, f, ensure_ascii=False, indent=2)

print("="*60)
print(f"FINAL AUDIT STATUS: {AUDIT_STATUS}")
print(f"NEXT ACTION:        {NEXT_ACTION}")
print("="*60)
print(f"\n저장 경로: {OUTPUT_DIR}")
print(f"  - FINAL_AUDIT_SUMMARY.md")
print(f"  - FINAL_AUDIT_SUMMARY.json")
print(f"  - audit_config.json")
print()
print(md_text)

FINAL AUDIT STATUS: AUDIT_FAILED
NEXT ACTION:        RECOMPUTE_M0_REQUIRED

저장 경로: /content/drive/MyDrive/Colab Notebooks/field_iis3dwb/audit_outputs/20260725_021125
  - FINAL_AUDIT_SUMMARY.md
  - FINAL_AUDIT_SUMMARY.json
  - audit_config.json

# FINAL AUDIT SUMMARY — M0_AUDIT_v1

생성 시각: 2026-07-25 04:02:55

---

## 1. Audit Status

```
AUDIT_FAILED
```

## 2. Dataset Structure

| Split | 실제 수 | 기대 수 | 일치 |
|:---|---:|---:|---:|
| LEARNING  | 6  | 6  | True |
| TEST      | 11  | 11  | True |
| FULL_TEST | 11  | 11  | True |
| **TOTAL** | **28** | **28** | **True** |

## 3. Duplicate-Name Root Cause

기존 M0 코드는 전체 경로가 아니라 `bd.name`을 결과 ID로 사용했기 때문에
`TEST/BearingX_Y`와 `FULL_TEST/BearingX_Y`가 동일 이름으로 표시됐다.
이는 데이터 삭제나 병합이 아니며, basename 사용에 따른 표기상 중복이다.

## 4. TEST/FULL_TEST Relation

| bearing | Test N | Full Test N | relation | hash ratio | evidence |
|:---|---:|---:|:---|---:|:---|
| Bearing1_3 | 590 | 2375 | TEST_IS_PREFIX_OF_FULL_TEST | 1.0 | N/A |
| Bearing1_4 | 1139 | 1428 | DIFFERENT